In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry
import os

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 16.0678,
	"longitude": 108.2208,
	"start_date": "2021-01-01",
	"end_date": "2026-05-25",
	"hourly": ["wind_speed_10m", "temperature_2m", "cloud_cover", "relative_humidity_2m", "surface_pressure", "apparent_temperature", "precipitation", "vapour_pressure_deficit"],
	"timezone": "Asia/Bangkok",
	"fbclid": "IwY2xjawSAxltleHRuA2FlbQIxMABicmlkETFORndIV0FvUUNRVEhJMTFpc3J0YwZhcHBfaWQQMjIyMDM5MTc4ODIwMDg5MgABHjUURbiYEDhgAq8yyKj-uqX2O28BidZKMjzoM12gWZvpl5BdTNZL9jNf_c6m_aem_Nd1pYWsZuDSn2axOcis1kw",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
hourly_temperature_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(2).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(3).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(4).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(5).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(6).ValuesAsNumpy()
hourly_vapour_pressure_deficit = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["vapour_pressure_deficit"] = hourly_vapour_pressure_deficit

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe.head(10))


Coordinates: 16.063268661499023°N 108.23863983154297°E
Elevation: 11.0 m asl
Timezone: b'Asia/Bangkok'b'GMT+7'
Timezone difference to GMT+0: 25200s

Hourly data
                        date  wind_speed_10m  temperature_2m  cloud_cover  \
0 2021-01-01 00:00:00+07:00       26.144030       19.500000        100.0   
1 2021-01-01 01:00:00+07:00       24.192429       19.500000        100.0   
2 2021-01-01 02:00:00+07:00       23.565569       19.350000         99.0   
3 2021-01-01 03:00:00+07:00       24.192429       19.200001         91.0   
4 2021-01-01 04:00:00+07:00       23.838959       19.250000        100.0   
5 2021-01-01 05:00:00+07:00       27.722336       19.100000         84.0   
6 2021-01-01 06:00:00+07:00       26.596178       19.150000         64.0   
7 2021-01-01 07:00:00+07:00       25.922499       19.549999         55.0   
8 2021-01-01 08:00:00+07:00       26.208397       19.700001         89.0   
9 2021-01-01 09:00:00+07:00       27.169865       19.950001         57.0   

 

In [3]:
test_hours = 24 * 7  
train_df = hourly_dataframe.iloc[:-test_hours]
test_df = hourly_dataframe.iloc[-test_hours:]

In [4]:
output_dir = "raw_data"
os.makedirs(output_dir, exist_ok=True)

file_path_train = os.path.join(output_dir, "raw_data_train.csv")
file_path_test = os.path.join(output_dir, "raw_data_test.csv")

train_df.to_csv(file_path_train, index=False, encoding="utf-8-sig")
test_df.to_csv(file_path_test, index=False, encoding="utf-8-sig")

print(f"\nXuất dữ liệu thành công! Các file đã được lưu tại thư mục '{output_dir}':")
print(f" ├── Dữ liệu Train: raw_data_train.csv ({len(train_df)} dòng)")
print(f" └── Dữ liệu Test:  raw_data_test.csv ({len(test_df)} dòng)")


Xuất dữ liệu thành công! Các file đã được lưu tại thư mục 'raw_data':
 ├── Dữ liệu Train: raw_data_train.csv (47136 dòng)
 └── Dữ liệu Test:  raw_data_test.csv (168 dòng)
